## Colab session configuration

Connect to google cloud runtime and use GPU.

## Reading hg38 genome assembly 

Set parent directory.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import pathlib
from pathlib import Path
!pip install pyfaidx
from pyfaidx import Fasta

# Set data directory and base directory
base_dir = Path.cwd()

Download repo and the hg38 fasta zipped file.

In [4]:
# Download repo to access scripts
!git clone https://github.com/eddykang06/mini-gLM.git
&cd mini-gLM

# Add repo scripts to path
sys.path.append("/content/mini-gLM-src")

# Download zipped human genome files
!mkdir -p data
!curl -L -C - "https://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz" -o data/hg38.fa.gz
!curl -L -C - "https://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes" -o data/hg38.chrom.sizes
!gunzip -c data/hg38.fa.gz > data/hg38.fa # Unzip to fasta

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  938M  100  938M    0     0  63.9M      0  0:00:14  0:00:14 --:--:-- 60.9M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 11672  100 11672    0     0  93291      0 --:--:-- --:--:-- --:--:-- 94129


Sample random sections of the genome, weighted by chromosome length and within the specified sequence length range.

In [9]:
from glm_pipeline import sample_from_fasta

# Set seed
rng = np.random.default_rng(111)

# Data directory
data_dir = base_dir / "data"
fasta_file = data_dir / "hg38.fa"
chrom_size_file = data_dir / "hg38.chrom.sizes"

# Set parameters
n_seqs = 100
min_length = 20
max_length = 500

# Get pretraining data
pretraining = sample_from_fasta(
    fasta_file = fasta_file,
    chrom_size_file = chrom_size_file,
    n_seqs = n_seqs,
    min_length = min_length,
    max_length = max_length
)

## Tokenizer

Steps to build tokenizer:
- Break corpus into words (["A", "C", "G", "T"])
- BPE algorithm

In [59]:
def find_all_adjacent_pairs(
        token_list: list
) -> dict:
    """
    Given a list of characters, return a dictionary with keys as unique adjacent character tuples and values as the # of times 
    that adjacent pair appears in the list

    Ex: ["A", "C", "G"] -> {("A", "C"): 1, ("C", "G"): 1}
    """
    all_pairs = []

    for i in range(len(token_list) - 1):
        pair = "".join(token_list[i:i+2])
        all_pairs.append(pair)
    unique_pairs, counts = np.unique_counts(all_pairs)
    unique_pair_counts = {tuple(pair): int(count) for pair, count in zip(unique_pairs, counts)}

    return unique_pair_counts

def merge_token_pairs(
        token_list: list, 
        token_pair: tuple
) -> list:
    """
    Given a list of tokens and a tuple of token pairs, return a list where all instances of the adjacent token pair have been merged

    Ex: ["A", "C", "G", "T"], ("A", "C") -> ["AC", "G", "T"]
    """
    merged_list = []
    counter = 0

    while counter < len(token_list):
        pair = tuple(token_list[counter:counter+2])
        if pair == token_pair:
            merged = "".join(pair)
            merged_list.append(merged)
            counter += 2
        else:
            old_token = pair[0]
            merged_list.append(old_token)
            counter += 1

    return merged_list

Quick BPE implementation.

In [ ]:
# Set seed for random selection
seed = 111
rng = np.random.default_rng(seed)

# Sample corpus
corpus = [
    "ACGTACTACATACTAA",
    "ACGTACA",
    "AATTG"
]

# Initial vocab
vocab = ["A", "C", "G", "T"]

# Set size of interest
final_size = 10

# Store merge rules for future tokenization
merge_rules = {}

# In iteration 1, simple split each sequence into list (treat each sequence as a word)
split_corpus = [list(seq) for seq in corpus]

# Iterate until vocab reaches the specified length
while len(vocab) < final_size:

    # Dictionary to store the counts for each pair seen in the corpus
    pair_counts_all = {}

    # For each sequence in the corpus, find the unique pairs and store them a lists
    for seq in split_corpus:

        # Find counts of all adjacent pairs in sequence and store as a dictionary
        pair_counts_seq = find_all_adjacent_pairs(seq)

        for pair, count in pair_counts_seq.items():
            
            # Add the key as a new pair if not yet seen, otherwise simply add to the counts
            pair_counts_all[pair] = pair_counts_all.get(pair, 0) + count
 
    # Store pairs and counts
    pairs = list(pair_counts_all.keys())
    counts = np.array(list(pair_counts_all.values()))

    # Find pair(s) with the highest counts
    best_idx = np.argwhere(counts == counts.max()).ravel()

    # If the new token already exists in the vocabulary, then select the next most common pair

    # 

    
    # If multiple pairs are found, then randomly select one
    if len(best_idx) > 1:
        random_idx = np.random.choice(best_idx)
        best_pair = pairs[random_idx]
    
    else:
        best_idx = best_idx[0]
        best_pair = pairs[best_idx]
    
    # Join all instances of the pair in split_corpus
    split_corpus = [merge_token_pairs(seq, best_pair) for seq in split_corpus]
    print(split_corpus)

    # Add new token to vocbaulary if its not redundant
    new_token = "".join(best_pair)
    if new_token not in vocab:
        vocab.append(new_token)
    else:
        break

    # Store merge rule
    merge_rule = {best_pair: new_token}
    merge_rules.update(merge_rule)

print(vocab)
print(merge_rules)

[['AC', 'G', 'T', 'AC', 'T', 'AC', 'A', 'T', 'AC', 'T', 'A', 'A'], ['AC', 'G', 'T', 'AC', 'A'], ['A', 'A', 'T', 'T', 'G']]
[['AC', 'G', 'T', 'AC', 'T', 'AC', 'A', 'T', 'AC', 'T', 'A', 'A'], ['AC', 'G', 'T', 'AC', 'A'], ['A', 'A', 'T', 'T', 'G']]
[['AC', 'G', 'T', 'AC', 'T', 'AC', 'A', 'T', 'AC', 'T', 'A', 'A'], ['AC', 'G', 'T', 'AC', 'A'], ['A', 'A', 'T', 'T', 'G']]
['A', 'C', 'G', 'T', 'AC', 'TAC']
{('A', 'C'): 'AC', ('T', 'A', 'C'): 'TAC'}


In [ ]:
def seqs_to_corpus():

    
    return corpus



class BPE_Tokenizer():
    def __init__(self, vocab, vocab_size):
        self.vocab = ["A", "C", "G", "T"]
        self.vocab_size = 4
    
    # Fit tokenizer on list of sequences
    def fit(self, seqs, desired_size):

    
    # Tokenize a given sequence
    def tokenize(self, seqs):




In [ ]:
class object:
    def __init__(self, stat):
        self.stat = stat



Reference: https://huggingface.co/learn/llm-course/en/chapter6/5

## Model

In [ ]:
!pip install accelerate
!pip install datasets
!pip install transformers
!pip install torch
!pip install flash-attn

import accelerate
import flash_attn
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

torch.backends.cudnn.benchmark=True

Implement a sparse attention transformer.

Implement the model with 10 transformer blocks.

In [ ]:
class miniglm(nn.Module):

    def __init__(self):